# Experimento 1 (EXP-BD-01): Formato de almacenamiento
## CSV vs. Parquet vs. Delta en Apache Spark (Databricks)
**Curso:** Base de Datos Avanzadas y Big Data · **Grupo 02** · UPN · Período 2026-2
**Dataset:** DataCo Supply Chain (capa Silver de la solución Medallion)

---
### 1. Objetivo
Medir cómo cambia el rendimiento de la solución Big Data al **guardar los mismos datos en distintos formatos de almacenamiento** (CSV, Parquet y Delta), a medida que crece el volumen.

**Objetivos específicos**
- Comparar el **espacio en disco** y el **tiempo de escritura** de cada formato.
- Comparar el **tiempo de lectura completa** y de las consultas de negocio **Q1–Q4** sobre cada formato.
- Comparar una **consulta selectiva** (con filtro) para ver el efecto de leer solo lo necesario.
- Determinar qué formato conviene para la capa de consumo de la arquitectura.

### 2. Hipótesis
- **H1.** Parquet y Delta ocupan menos espacio que CSV.
- **H2.** Parquet y Delta responden Q1–Q4 en menos tiempo que CSV, porque son formatos **columnares** y solo leen las columnas necesarias.
- **H3.** Delta tiene un rendimiento similar al de Parquet, con una pequeña sobrecarga por su registro de transacciones.

> Las hipótesis se **verifican con los datos medidos** en la Parte 5; pueden cumplirse o no.

### 3. Variables
| Tipo | Variable |
|---|---|
| Independiente | Formato de almacenamiento: CSV (esquema explícito), CSV (inferSchema), Parquet, Delta |
| Independiente | Volumen de datos: 1×, 4× y 8× el dataset original |
| Dependiente | Tamaño en disco (MB) y número de archivos |
| Dependiente | Tiempo de escritura (s) |
| Dependiente | Tiempo de lectura completa y de las consultas Q1–Q4 y Q5 (s), con mediana, desviación y CV |

### 4. Configuración y condiciones de prueba
| Elemento | Valor |
|---|---|
| Plataforma | Databricks Free Edition, cómputo serverless |
| Motor | Apache Spark (versión registrada en la Parte 1) |
| Dataset de entrada | `silver_supply_chain` (180 519 filas, ya limpio y tipado) |
| Escalas | 1×, 4×, 8× (réplica con `crossJoin`, claves desplazadas para mantener la unicidad) |
| Formatos | CSV con esquema explícito · CSV con `inferSchema` · Parquet · Delta |
| Consultas | Lectura completa, Q1, Q2, Q3, Q4 y Q5 (consulta selectiva) |
| Repeticiones | 1 corrida de calentamiento descartada + **5 repeticiones** por consulta |
| Estadísticos | Mediana, media, desviación estándar y CV |
| Umbral de variabilidad | CV ≤ 10 % (≤ 15 % si la mediana es < 1 s) |
| Qué mide el tiempo | **Extremo a extremo**: construir el DataFrame desde el almacenamiento + ejecutar la consulta + traer el resultado |

**Condiciones y limitaciones (deben ir en el informe)**
- El cómputo serverless es compartido y **no se controlan nodos ni memoria**; por eso se usa calentamiento y repeticiones.
- El tiempo de **escritura se mide una sola vez** por formato y escala, por lo que no tiene CV.
- CSV y Parquet se guardan en un **Volume**; Delta se guarda como **tabla administrada**. Todos usan el mismo almacenamiento de la nube.
- El tamaño cuenta solo los **archivos de datos** (`part-*` en CSV y Parquet; `sizeInBytes` en Delta), sin el registro de transacciones de Delta.
- CSV se compara con **esquema explícito** para aislar el costo del formato; `inferSchema` se mide aparte para mostrar el costo de inferir tipos.
- Las escalas son réplicas del dataset: sirven para medir costo de procesamiento, no crecimiento real del negocio.

### 5. Procedimiento
1. **Parte 1.** Definir parámetros y registrar fecha y versión de Spark.
2. **Parte 2.** Crear las tablas base por escala a partir de Silver (1×, 4×, 8×).
3. **Parte 3.** Escribir cada escala en CSV, Parquet y Delta, midiendo tiempo y tamaño.
4. **Parte 4.** Medir lectura completa y consultas Q1–Q4 y Q5 en cada formato y escala (calentamiento + 5 repeticiones).
5. **Parte 5.** Presentar resultados, compararlos y verificar las hipótesis.
6. **Parte 6.** Guardar resultados como tablas (evidencia) y limpiar los archivos de prueba.

## Parte 1. Parámetros y entorno
> **Antes de empezar:** debe existir la tabla Silver. Ejecute primero el notebook Medallion hasta la Parte 4. Si usa el notebook anterior, cambie `TABLA_ORIGEN` a `tb_supply_chain`.
>
> **Tiempo de ejecución:** CSV a 8× es el caso más lento. Si demora demasiado, deje `ESCALAS = [1, 4]`.

In [0]:
import time, datetime, statistics
import pandas as pd
from pyspark.sql import functions as F

# ---------- Parámetros del experimento
TABLA_ORIGEN = "silver_supply_chain"
ESCALAS = [1, 4, 8]
REPS = 5
BASE_DIR = "/Volumes/workspace/default/datacodatos/exp01"
FORMATOS = ["CSV (esquema explícito)", "CSV (inferSchema)", "Parquet", "Delta"]

meta = {"experimento": "EXP-BD-01 Formato de almacenamiento",
        "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
        "spark": spark.version,
        "compute": "Databricks Free Edition (serverless)",
        "escalas": str(ESCALAS), "repeticiones": REPS}
print(meta)

# ---------- Funciones de apoyo
def ruta(fmt, k):
    return f"{BASE_DIR}/{fmt}_{k}x"

def tam_dir(path):
    """Suma el tamaño (bytes) y cuenta los archivos de datos part-* de una carpeta."""
    total, n = 0, 0
    for f in dbutils.fs.ls(path):
        if f.isDir():
            b, c = tam_dir(f.path); total += b; n += c
        elif f.name.startswith("part-"):
            total += f.size; n += 1
    return total, n

def ver(pdf):
    """Muestra un DataFrame de pandas como tabla de Databricks (permite el + para graficar)."""
    display(spark.createDataFrame(pdf))

{'experimento': 'EXP-BD-01 Formato de almacenamiento', 'fecha': '2026-11-26T00:31:12', 'spark': '4.2.0', 'compute': 'Databricks Free Edition (serverless)', 'escalas': '[1, 4, 8]', 'repeticiones': 5}


## Parte 2. Crear los datos base por escala
Se crea una **tabla base por escala** (`exp01_base_Nx`). Así, el costo de generar la réplica queda fuera de los tiempos y los tres formatos se escriben desde exactamente los mismos datos.

In [0]:
origen = spark.table(TABLA_ORIGEN)
SCHEMA = origen.schema                     # esquema explícito para leer el CSV

def crear_escala(k):
    if k == 1:
        return origen
    return (origen.crossJoin(spark.range(k).withColumnRenamed("id", "replica"))
                  .withColumn("order_id",      F.col("order_id")      + F.col("replica") * F.lit(1000000))
                  .withColumn("order_item_id", F.col("order_item_id") + F.col("replica") * F.lit(1000000))
                  .drop("replica"))

for k in ESCALAS:
    crear_escala(k).write.mode("overwrite").saveAsTable(f"exp01_base_{k}x")

FILAS = {k: spark.table(f"exp01_base_{k}x").count() for k in ESCALAS}
display(spark.createDataFrame([(f"{k}x", FILAS[k], len(SCHEMA.fields)) for k in ESCALAS],
                               schema="escala STRING, filas LONG, columnas INT"))

escala,filas,columnas
1x,180519,54
4x,722076,54
8x,1444152,54


## Parte 3. Escritura en cada formato (tiempo y tamaño)
Cada escala se escribe en **CSV** (con encabezado), **Parquet** y **Delta** desde la misma tabla base. Se mide el tiempo de escritura (1 vez) y el tamaño en disco.

In [0]:
escritura = []
for k in ESCALAS:
    base = spark.table(f"exp01_base_{k}x")
    for fmt in ["csv", "parquet", "delta"]:
        t0 = time.perf_counter()
        if fmt == "csv":
            base.write.mode("overwrite").option("header", "true").csv(ruta("csv", k))
        elif fmt == "parquet":
            base.write.mode("overwrite").parquet(ruta("parquet", k))
        else:
            base.write.mode("overwrite").saveAsTable(f"exp01_delta_{k}x")
        seg = time.perf_counter() - t0

        if fmt == "delta":
            d = spark.sql(f"DESCRIBE DETAIL exp01_delta_{k}x").select("sizeInBytes", "numFiles").collect()[0]
            nbytes, narch = d["sizeInBytes"], d["numFiles"]
        else:
            nbytes, narch = tam_dir(ruta(fmt, k))

        escritura.append({"escala": f"{k}x", "filas": FILAS[k], "formato": {"csv": "CSV", "parquet": "Parquet", "delta": "Delta"}[fmt],
                          "tamano_mb": round(nbytes / 1e6, 2), "archivos": int(narch), "escritura_s": round(seg, 2)})
        print(f"{k}x {fmt}: {seg:.1f} s | {nbytes/1e6:.1f} MB")

alm = pd.DataFrame(escritura)
csv_mb = alm[alm.formato == "CSV"].set_index("escala")["tamano_mb"]
alm["tamano_vs_csv"] = (alm["tamano_mb"] / alm["escala"].map(csv_mb)).round(3)
alm["reduccion_vs_csv_pct"] = ((1 - alm["tamano_vs_csv"]) * 100).round(1)
alm["throughput_escritura_filas_s"] = (alm["filas"] / alm["escritura_s"]).round(0)

1x csv: 5.8 s | 97.6 MB
1x parquet: 1.7 s | 9.5 MB
1x delta: 3.8 s | 8.1 MB
4x csv: 9.1 s | 392.3 MB
4x parquet: 2.2 s | 24.8 MB
4x delta: 4.0 s | 12.7 MB
8x csv: 15.7 s | 785.2 MB
8x parquet: 2.8 s | 37.6 MB
8x delta: 4.8 s | 16.7 MB


## Parte 4. Medición de lectura y consultas
Se mide cada consulta con **1 corrida de calentamiento (descartada) + 5 repeticiones**. En cada repetición el DataFrame se **vuelve a construir desde el almacenamiento**, de modo que el tiempo incluye leer el formato (y, en `inferSchema`, inferir los tipos).

| Consulta | Qué prueba |
|---|---|
| Lectura completa | Leer **todas** las columnas (suma del hash de cada fila) |
| Q1 | Indicadores globales (pocas columnas) |
| Q2 | Retrasos por modo de envío |
| Q3 | Retrasos por región y categoría |
| Q4 | Retrasos y ganancia por año y mes |
| Q5 | Consulta **selectiva**: filtro por mercado y modo de envío |

In [0]:
# Consultas de negocio (las mismas Q1–Q4 de la solución)
def q1(t):
    return t.agg(F.count("*").alias("total_entregas"), F.sum("late_delivery_risk").alias("entregas_tardias"),
                 F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias"),
                 F.round(F.avg("days_for_shipping_real"), 2).alias("dias_reales_prom"),
                 F.round(F.avg("days_for_shipment_scheduled"), 2).alias("dias_prog_prom"))

def q2(t):
    return (t.groupBy("shipping_mode")
             .agg(F.count("*").alias("entregas"), F.round(F.avg("late_delivery_risk") * 100, 2).alias("pct_tardias"))
             .orderBy(F.desc("pct_tardias")))

def q3(t):
    return (t.groupBy("order_region", "category_name")
             .agg(F.sum("late_delivery_risk").alias("entregas_tardias")).orderBy(F.desc("entregas_tardias")))

def q4(t):
    return (t.groupBy("order_year", "order_month")
             .agg(F.sum("late_delivery_risk").alias("entregas_tardias"),
                  F.round(F.sum("benefit_per_order"), 2).alias("ganancia_total"))
             .orderBy("order_year", "order_month"))

def q5(t):
    return (t.filter((F.col("market") == "LATAM") & (F.col("shipping_mode") == "First Class"))
             .agg(F.count("*").alias("entregas"), F.sum("late_delivery_risk").alias("tardias")))

def lectura_completa(t):
    return t.agg(F.sum(F.hash(*t.columns)).alias("checksum"))

CONSULTAS = {"Lectura completa": lectura_completa, "Q1": q1, "Q2": q2, "Q3": q3, "Q4": q4, "Q5 selectiva": q5}

# Cómo se construye el DataFrame desde cada formato
LECTORES = {
  "CSV (esquema explícito)": lambda k: spark.read.option("header", "true").schema(SCHEMA).csv(ruta("csv", k)),
  "CSV (inferSchema)":       lambda k: spark.read.option("header", "true").option("inferSchema", "true").csv(ruta("csv", k)),
  "Parquet":                 lambda k: spark.read.parquet(ruta("parquet", k)),
  "Delta":                   lambda k: spark.table(f"exp01_delta_{k}x"),
}

In [0]:
def medir(fn, lector, k, reps=REPS):
    fn(lector(k)).collect()                      # calentamiento (descartado)
    tiempos = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn(lector(k)).collect()                  # construir DataFrame + ejecutar + traer resultado
        tiempos.append(time.perf_counter() - t0)
    return tiempos

resultados = []
for k in ESCALAS:
    for fmt in FORMATOS:
        for qn, fn in CONSULTAS.items():
            ts = medir(fn, LECTORES[fmt], k)
            med, media, sd = statistics.median(ts), statistics.mean(ts), statistics.stdev(ts)
            cv = sd / media * 100
            resultados.append({"escala": f"{k}x", "filas": FILAS[k], "formato": fmt, "consulta": qn,
                               "mediana_s": round(med, 3), "media_s": round(media, 3), "desv_s": round(sd, 3),
                               "cv_pct": round(cv, 1), "throughput_filas_s": round(FILAS[k] / med),
                               "cumple_CV": cv <= (15 if med < 1 else 10),
                               "tiempos_s": [round(x, 3) for x in ts]})
        print(f"Listo: {k}x · {fmt}")

df_res = pd.DataFrame(resultados)
print("Mediciones:", len(df_res))

Listo: 1x · CSV (esquema explícito)
Listo: 1x · CSV (inferSchema)
Listo: 1x · Parquet
Listo: 1x · Delta
Listo: 4x · CSV (esquema explícito)
Listo: 4x · CSV (inferSchema)
Listo: 4x · Parquet
Listo: 4x · Delta
Listo: 8x · CSV (esquema explícito)
Listo: 8x · CSV (inferSchema)
Listo: 8x · Parquet
Listo: 8x · Delta
Mediciones: 72


## Parte 5. Resultados, comparación y análisis
Cada tabla queda lista para graficar: clic en el **+** junto a *Table* → **Visualization**.

### 5.1 Espacio en disco y tiempo de escritura
**Gráficos sugeridos (+):**
- Barras agrupadas · X = `escala` · Y = `tamano_mb` · Agrupar por = `formato`.
- Barras agrupadas · X = `escala` · Y = `escritura_s` · Agrupar por = `formato`.

In [0]:
ver(alm)

escala,filas,formato,tamano_mb,archivos,escritura_s,tamano_vs_csv,reduccion_vs_csv_pct,throughput_escritura_filas_s
1x,180519,CSV,97.56,1,5.82,1.0,0.0,31017.0
1x,180519,Parquet,9.49,1,1.74,0.097,90.3,103747.0
1x,180519,Delta,8.14,1,3.8,0.083,91.7,47505.0
4x,722076,CSV,392.26,1,9.06,1.0,0.0,79699.0
4x,722076,Parquet,24.85,1,2.16,0.063,93.7,334294.0
4x,722076,Delta,12.67,1,4.03,0.032,96.8,179175.0
8x,1444152,CSV,785.2,1,15.66,1.0,0.0,92219.0
8x,1444152,Parquet,37.59,1,2.77,0.048,95.2,521355.0
8x,1444152,Delta,16.68,1,4.75,0.021,97.9,304032.0


Databricks visualization. Run in Databricks to view.

### 5.2 Tiempos de lectura y consultas (detalle de las 5 repeticiones)
**Gráfico sugerido (+):** Barras agrupadas · X = `consulta` · Y = `mediana_s` · Agrupar por = `formato` (filtre una escala a la vez).

In [0]:
ver(df_res.drop(columns=["tiempos_s"]))

escala,filas,formato,consulta,mediana_s,media_s,desv_s,cv_pct,throughput_filas_s,cumple_CV
1x,180519,CSV (esquema explícito),Lectura completa,1.267,1.246,0.081,6.5,142480,true
1x,180519,CSV (esquema explícito),Q1,0.989,0.974,0.041,4.2,182492,true
1x,180519,CSV (esquema explícito),Q2,0.963,0.964,0.012,1.3,187464,true
1x,180519,CSV (esquema explícito),Q3,0.988,1.027,0.112,10.9,182795,true
1x,180519,CSV (esquema explícito),Q4,0.985,0.982,0.017,1.7,183271,true
1x,180519,CSV (esquema explícito),Q5 selectiva,0.953,0.986,0.109,11.1,189419,true
1x,180519,CSV (inferSchema),Lectura completa,2.036,2.024,0.052,2.6,88677,true
1x,180519,CSV (inferSchema),Q1,1.67,1.694,0.059,3.5,108088,true
1x,180519,CSV (inferSchema),Q2,1.677,1.748,0.151,8.6,107633,true
1x,180519,CSV (inferSchema),Q3,1.741,1.713,0.049,2.9,103674,true


Databricks visualization. Run in Databricks to view.

In [0]:
# Las 5 mediciones de cada caso (evidencia de repeticiones)
evid = df_res[["escala", "formato", "consulta", "tiempos_s"]].copy()
evid["tiempos_s"] = evid["tiempos_s"].astype(str)
ver(evid)

escala,formato,consulta,tiempos_s
1x,CSV (esquema explícito),Lectura completa,"[1.31, 1.267, 1.224, 1.313, 1.116]"
1x,CSV (esquema explícito),Q1,"[0.949, 1.02, 0.995, 0.989, 0.916]"
1x,CSV (esquema explícito),Q2,"[0.978, 0.972, 0.945, 0.963, 0.962]"
1x,CSV (esquema explícito),Q3,"[1.195, 1.081, 0.988, 0.954, 0.918]"
1x,CSV (esquema explícito),Q4,"[0.959, 0.974, 0.985, 1.005, 0.986]"
1x,CSV (esquema explícito),Q5 selectiva,"[0.901, 1.177, 0.953, 0.939, 0.961]"
1x,CSV (inferSchema),Lectura completa,"[2.077, 2.036, 2.011, 2.056, 1.942]"
1x,CSV (inferSchema),Q1,"[1.789, 1.66, 1.713, 1.67, 1.639]"
1x,CSV (inferSchema),Q2,"[1.809, 1.677, 1.983, 1.658, 1.612]"
1x,CSV (inferSchema),Q3,"[1.741, 1.642, 1.742, 1.757, 1.682]"


### 5.3 Comparación contra CSV
`csv_infer_vs_csv_explicito`, `parquet_vs_csv_explicito` y `delta_vs_csv_explicito` indican **cuántas veces más rápido** es cada formato respecto a CSV con esquema explícito (**> 1 = más rápido**, **< 1 = más lento**).

**Gráfico sugerido (+):** Barras agrupadas · X = `consulta` · Y = `csv_explicito`, `parquet` y `delta` (tiempos en segundos), filtrando por escala.

In [0]:
claves = {"CSV (esquema explícito)": "csv_explicito", "CSV (inferSchema)": "csv_infer", "Parquet": "parquet", "Delta": "delta"}

ancho = (df_res.pivot_table(index=["escala", "filas", "consulta"], columns="formato", values="mediana_s")
               .reset_index().rename(columns=claves))
ancho.columns.name = None
for f in ["csv_infer", "parquet", "delta"]:
    ancho[f"{f}_vs_csv_explicito"] = (ancho["csv_explicito"] / ancho[f]).round(2)

orden = {q: i for i, q in enumerate(CONSULTAS)}
ancho = (ancho.assign(_o=ancho["consulta"].map(orden)).sort_values(["filas", "_o"]).drop(columns="_o").reset_index(drop=True))
ver(ancho)

escala,filas,consulta,csv_explicito,csv_infer,delta,parquet,csv_infer_vs_csv_explicito,parquet_vs_csv_explicito,delta_vs_csv_explicito
1x,180519,Lectura completa,1.267,2.036,0.68,0.944,0.62,1.34,1.86
1x,180519,Q1,0.989,1.67,0.439,0.838,0.59,1.18,2.25
1x,180519,Q2,0.963,1.677,0.422,0.818,0.57,1.18,2.28
1x,180519,Q3,0.988,1.741,0.414,0.803,0.57,1.23,2.39
1x,180519,Q4,0.985,1.735,0.391,0.819,0.57,1.2,2.52
1x,180519,Q5 selectiva,0.953,1.691,0.44,0.788,0.56,1.21,2.17
4x,722076,Lectura completa,1.643,3.252,0.774,1.104,0.51,1.49,2.12
4x,722076,Q1,1.21,2.728,0.463,0.728,0.44,1.66,2.61
4x,722076,Q2,1.214,2.855,0.404,0.804,0.43,1.51,3.0
4x,722076,Q3,1.229,2.66,0.41,0.779,0.46,1.58,3.0


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### 5.4 Tiempo promedio de Q1–Q4 por formato y volumen
**Gráfico sugerido (+):** Líneas · X = `filas` · Y = `mediana_Q1_Q4_s` · Series = `formato`. Muestra cómo crece el tiempo con el volumen en cada formato.

In [0]:
resumen = (df_res[df_res.consulta.isin(["Q1", "Q2", "Q3", "Q4"])]
           .groupby(["escala", "filas", "formato"], as_index=False)
           .agg(mediana_Q1_Q4_s=("mediana_s", "median"), peor_Q_s=("mediana_s", "max"), cv_max_pct=("cv_pct", "max"))
           .sort_values(["filas", "mediana_Q1_Q4_s"]).reset_index(drop=True))
ver(resumen)

escala,filas,formato,mediana_Q1_Q4_s,peor_Q_s,cv_max_pct
1x,180519,Delta,0.418,0.439,8.9
1x,180519,Parquet,0.8185,0.838,9.0
1x,180519,CSV (esquema explícito),0.9864999999999999,0.989,10.9
1x,180519,CSV (inferSchema),1.706,1.741,8.6
4x,722076,Delta,0.40700000000000003,0.463,9.1
4x,722076,Parquet,0.7915000000000001,0.809,6.7
4x,722076,CSV (esquema explícito),1.218,1.229,5.5
4x,722076,CSV (inferSchema),2.694,2.855,6.7
8x,1444152,Delta,0.388,0.422,6.2
8x,1444152,Parquet,0.7464999999999999,0.813,5.4


Databricks visualization. Run in Databricks to view.

### 5.5 Verificación de hipótesis
La tabla compara los resultados medidos con H1, H2 y H3. Cada columna dice `True` o `False` según los **datos de su ejecución**.

In [0]:
piv = resumen.pivot_table(index=["escala", "filas"], columns="formato", values="mediana_Q1_Q4_s").reset_index()
piv.columns.name = None
tam = alm.pivot(index="escala", columns="formato", values="tamano_mb")

verif = pd.DataFrame({
    "escala": piv["escala"], "filas": piv["filas"],
    "csv_Q1_Q4_s": piv["CSV (esquema explícito)"], "parquet_Q1_Q4_s": piv["Parquet"], "delta_Q1_Q4_s": piv["Delta"]})
verif["H1_parquet_ocupa_menos"] = verif["escala"].map(lambda e: bool(tam.loc[e, "Parquet"] < tam.loc[e, "CSV"]))
verif["H1_delta_ocupa_menos"]   = verif["escala"].map(lambda e: bool(tam.loc[e, "Delta"] < tam.loc[e, "CSV"]))
verif["H2_parquet_mas_rapido"]  = verif["parquet_Q1_Q4_s"] < verif["csv_Q1_Q4_s"]
verif["H2_delta_mas_rapido"]    = verif["delta_Q1_Q4_s"] < verif["csv_Q1_Q4_s"]
verif["H3_delta_vs_parquet"]    = (verif["delta_Q1_Q4_s"] / verif["parquet_Q1_Q4_s"]).round(2)   # ≈ 1 = similares
ver(verif)

print("Casos que superan el umbral de variabilidad (cumple_CV = False):", int((~df_res["cumple_CV"]).sum()), "de", len(df_res))

escala,filas,csv_Q1_Q4_s,parquet_Q1_Q4_s,delta_Q1_Q4_s,H1_parquet_ocupa_menos,H1_delta_ocupa_menos,H2_parquet_mas_rapido,H2_delta_mas_rapido,H3_delta_vs_parquet
1x,180519,0.9864999999999999,0.8185,0.418,true,true,true,true,0.51
4x,722076,1.218,0.7915000000000001,0.40700000000000003,true,true,true,true,0.51
8x,1444152,1.688,0.7464999999999999,0.388,true,true,true,true,0.52


Casos que superan el umbral de variabilidad (cumple_CV = False): 0 de 72


### 5.6 Análisis y conclusiones
*Complete con lo observado en sus tablas y gráficos.* Preguntas guía:

1. **Espacio:** ¿cuánto menos ocupa Parquet y Delta respecto a CSV en cada escala (`reduccion_vs_csv_pct`)? ¿Se cumple H1?

Espacio (H1: se cumple). Parquet y Delta ocupan mucho menos que CSV en todas las escalas.
La reducción crece con el volumen en parte porque las escalas son réplicas del mismo dato: los formatos columnares comprimen muy bien las filas repetidas. En datos reales la reducción sería menor.

2. **Escritura:** ¿qué formato es más lento o más rápido de escribir? ¿Vale la pena el costo de escribir en un formato columnar si se lee muchas veces?

Parquet fue el más rápido en todas las escalas (1.74 / 2.16 / 2.77 s), Delta quedó en medio (3.80 / 4.03 / 4.75 s) y CSV fue el más lento (5.82 / 9.06 / 15.66 s). Con 8× más datos, CSV tardó 2.7 veces más, Parquet 1.6 y Delta 1.25. Delta cuesta algo más que Parquet por su registro de transacciones, pero lo compensa en lectura (ver punto 4).

3. **Lectura completa vs. consultas:** ¿la ventaja de Parquet/Delta es mayor en Q1 (pocas columnas) que en la lectura completa (todas las columnas)? Esto confirma o no el beneficio del formato columnar.

La ventaja de los formatos columnares es mayor cuando se leen pocas columnas. A 8×, Parquet es 1.94 veces más rápido que CSV en la lectura completa y 2.08 a 2.38 veces en Q1–Q4. Delta pasa de 2.62 veces (lectura completa) a 4.00–4.66 veces (Q1–Q4). Esto confirma el beneficio del formato columnar. A 1× el efecto en Parquet casi no se nota, porque dominan los costos fijos.

4. **Consulta selectiva (Q5):** ¿mejora más en Parquet/Delta que en CSV?

No mejoró más que las demás consultas: sus tiempos son similares a los de Q1–Q4 en los tres formatos. Cada formato generó un solo archivo y los datos no están ordenados ni particionados por mercado o modo de envío, así que Spark no tiene archivos que descartar. Esto abre la puerta a un siguiente experimento con particionado u OPTIMIZE + ZORDER.

5. **`inferSchema`:** ¿cuánto cuesta inferir los tipos frente al esquema explícito?

Inferir los tipos hizo la lectura 1.7 veces más lenta a 1×, 2.2 veces a 4× y 2.4 veces a 8× (Q1–Q4: 1.71 vs 0.99 s, 2.69 vs 1.22 s y 4.13 vs 1.69 s). El costo crece con el volumen porque la inferencia recorre los datos una vez más. Conclusión: siempre usar esquema explícito.

6. **Escalabilidad:** al pasar de 1× a 8×, ¿cómo crece el tiempo de cada formato? (ver 5.4)

Con 8 veces más datos, el tiempo de Q1–Q4 en CSV (esquema explícito) subió de 0.99 a 1.69 s, y en CSV con inferSchema de 1.71 a 4.13 s. En Parquet (0.82 → 0.75 s) y Delta (0.42 → 0.39 s) se mantuvo prácticamente plano. Eso indica que, hasta 1.4 millones de filas, en esos formatos domina el costo fijo (planificación, metadatos, comunicación con el cómputo serverless) y no el volumen. No se puede extrapolar a volúmenes mayores.

7. **Variabilidad:** ¿qué casos superan el umbral de CV y por qué (cómputo compartido, cachés)?

Ningún caso superó el umbral de CV (0 de 72). El CV promedio fue 4.9 % y el máximo 13.5 %, que está dentro del 15 % permitido para consultas menores a 1 s. La escritura se midió una sola vez, así que no tiene CV.

8. **Decisión:** ¿qué formato recomienda para las capas Silver y Gold? ¿Por qué?

Se cumplieron H1 y H2 y no se cumplió H3. Frente a CSV, los formatos columnares redujeron el espacio entre 90 % y 98 % y aceleraron las consultas hasta 2.4 veces (Parquet) y 4.7 veces (Delta) a 8× el volumen. Delta fue el mejor formato de lectura en las condiciones del experimento, aunque su escritura fue más lenta que la de Parquet. Hasta 1.4 millones de filas, Parquet y Delta casi no variaron con el volumen, a diferencia de CSV.

**Conclusión del experimento:**

Se cumplieron H1 y H2 y no se cumplió H3. Frente a CSV, los formatos columnares redujeron el espacio entre 90 % y 98 % y aceleraron las consultas hasta 2.4 veces (Parquet) y 4.7 veces (Delta) a 8× el volumen. Delta fue el mejor formato de lectura en las condiciones del experimento, aunque su escritura fue más lenta que la de Parquet. Hasta 1.4 millones de filas, Parquet y Delta casi no variaron con el volumen, a diferencia de CSV.

**Limitaciones:** cómputo serverless sin control de recursos; escritura medida una vez; réplicas del mismo dataset; CSV y Parquet en Volume y Delta como tabla administrada.

1. Cómputo serverless compartido y sin control de recursos.
2. La escritura se midió una sola vez.
3. Las escalas son réplicas, lo que exagera la compresión.
4. CSV y Parquet se leyeron desde un Volume y Delta como tabla del catálogo.
5. Cada formato generó un solo archivo, por lo que no se evaluó el efecto de saltar archivos.

## Parte 6. Evidencias y limpieza
### 6.1 Guardar los resultados como tablas (evidencia)

In [0]:
def con_meta(pdf):
    out = pdf.copy()
    out["fecha"], out["spark"] = meta["fecha"], meta["spark"]
    return out

det = df_res.copy(); det["tiempos_s"] = det["tiempos_s"].astype(str)

spark.createDataFrame(con_meta(alm)).write.mode("overwrite").saveAsTable("tb_exp01_almacenamiento")
spark.createDataFrame(con_meta(det)).write.mode("overwrite").saveAsTable("tb_exp01_resultados")
spark.createDataFrame(con_meta(ancho)).write.mode("overwrite").saveAsTable("tb_exp01_comparacion")
spark.createDataFrame(con_meta(verif)).write.mode("overwrite").saveAsTable("tb_exp01_hipotesis")
print("Guardadas: tb_exp01_almacenamiento, tb_exp01_resultados, tb_exp01_comparacion, tb_exp01_hipotesis")

Guardadas: tb_exp01_almacenamiento, tb_exp01_resultados, tb_exp01_comparacion, tb_exp01_hipotesis


In [0]:
# Archivos generados por el experimento (evidencia de la ejecución)
display(dbutils.fs.ls(BASE_DIR))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/datacodatos/exp01/csv_1x/,csv_1x/,0,1795654355931
dbfs:/Volumes/workspace/default/datacodatos/exp01/csv_4x/,csv_4x/,0,1795654355931
dbfs:/Volumes/workspace/default/datacodatos/exp01/csv_8x/,csv_8x/,0,1795654355931
dbfs:/Volumes/workspace/default/datacodatos/exp01/parquet_1x/,parquet_1x/,0,1795654355931
dbfs:/Volumes/workspace/default/datacodatos/exp01/parquet_4x/,parquet_4x/,0,1795654355931
dbfs:/Volumes/workspace/default/datacodatos/exp01/parquet_8x/,parquet_8x/,0,1795654355931


### 6.2 Lista de evidencias para el informe
| # | Evidencia | De dónde sale |
|---|---|---|
| 1 | Parámetros, fecha y versión de Spark | Parte 1 (salida de `meta`) |
| 2 | Tablas base por escala y número de filas | Parte 2 |
| 3 | Archivos generados en el Volume y tablas `exp01_*` en Catalog | Parte 6.1 y Catalog |
| 4 | Tabla de espacio y tiempo de escritura + gráficos de barras | 5.1 |
| 5 | Tabla de tiempos con mediana, desviación y CV | 5.2 |
| 6 | Las 5 mediciones de cada caso | 5.2 (segunda tabla) |
| 7 | Comparación contra CSV + gráfico | 5.3 |
| 8 | Gráfico de líneas tiempo vs. volumen | 5.4 |
| 9 | Verificación de hipótesis | 5.5 |
| 10 | Tablas `tb_exp01_*` guardadas | Parte 6.1 |

### 6.3 Limpieza
Los archivos y tablas del experimento ocupan almacenamiento. **Ejecute la celda solo después de guardar las evidencias** (capturas de pantalla y tablas `tb_exp01_*`).

In [0]:
# for k in ESCALAS:
#     spark.sql(f"DROP TABLE IF EXISTS exp01_base_{k}x")
#     spark.sql(f"DROP TABLE IF EXISTS exp01_delta_{k}x")
# dbutils.fs.rm(BASE_DIR, True)